# Sección 3 · Un VCF real: anatomía, calidad y decisiones

*Cómo leer un archivo de secuenciación real, encontrar sus problemas y justificar cada filtro*

## Objetivos de aprendizaje

Al terminar esta sección podrás:

1. **Identificar** las partes de un VCF (encabezado, columnas fijas, `FORMAT` y genotipos) y leer una línea completa.
2. **Convertir** un VCF en una matriz de dosis (0, 1, 2, `NA`) y comprobar que está alineada con sus metadatos.
3. **Construir** la ficha técnica del archivo: número de árboles y de SNP, datos faltantes, SNP sin variación y distribución de MAF.
4. **Reconocer** tres señales de alerta: exceso de heterocigotos (posibles parálogos), pérdida de heterocigotos por baja profundidad y **réplicas**.
5. **Aplicar** los umbrales de un estudio real y **medir** lo que cuestan.
6. **Justificar** por escrito una decisión de filtrado para cada alerta que encuentres.

## Los datos

Trabajamos con `datos_ejemplo/lenga_real_1782snp.vcf.gz`, un subconjunto del VCF publicado por Sekely et al. (2024) sobre *Nothofagus pumilio*:

- **207 muestras**: 200 árboles (10 por sitio, 20 sitios), **6 réplicas** y **1 muestra** que quedó casi sin datos.
- **1 782 SNP**, un SNP al azar por contig, en **1 782 contigs**.
- Cada genotipo trae `GT:AD:DP:GQ`; el `INFO` conserva los campos del archivo original.

Los contigs se llaman `NODE_…` o `chain_…`: son **nodos de un ensamblaje *de novo***, no cromosomas. La ficha completa del archivo está en `datos_ejemplo/LEEME.md`.

**Revisión de la tarea de la Sección 2:** el gráfico de He por población con intervalos, la frase de conclusión y las tres líneas de Limitaciones.

**Y la revisión de GATK HaplotypeCaller** (tarea de la Sección 0): qué comparamos entre llamadores y qué se pierde al trabajar sobre un ensamblaje *de novo* sin genoma de referencia.

> **Antes de empezar, anota dos predicciones:** ¿qué porcentaje de genotipos crees que falta en este archivo? ¿Y cuántos de los 1 782 SNP crees que **no varían** entre estos 207 árboles?

## Cómo está organizada

- **Parte demostrativa** (se hace junto al docente): anatomía del archivo, ficha técnica, tres señales de alerta y, ya con el archivo limpio, **diversidad por sitio (Ho, He, Fis) y PCA**.
- **Parte de ejercicios** (la haces tú): repites el análisis con variaciones y entregas una ficha técnica y tres alertas con su decisión.

In [ ]:
# ---- Preparación (ejecutar primero) ----
options(repr.plot.width = 8, repr.plot.height = 4.2, repr.plot.res = 110)
suppressPackageStartupMessages({ library(vcfR); library(ggplot2) })
theme_set(theme_minimal(base_size = 13))

VCF  <- "datos_ejemplo/lenga_real_1782snp.vcf.gz"
META <- "datos_ejemplo/lenga_real_muestras.csv"
# Si falla: abriste Jupyter desde otra carpeta; ábrelo desde la carpeta del repositorio.
stopifnot(file.exists(VCF), file.exists(META))

---
# Parte demostrativa

## 1. Un VCF es un archivo de texto

Antes de usar cualquier paquete, mirémoslo como lo que es:

- Las líneas que empiezan con `##` son el **encabezado**: describen el archivo (versión, contigs, campos de `INFO` y de `FORMAT`).
- La línea que empieza con `#CHROM` nombra las columnas.
- Cada línea siguiente es **un SNP**; desde la décima columna hay **un árbol por columna**.

Está comprimido (`.gz`), así que se lee con `gzfile()`.

*Predice:* ¿cuántas líneas de encabezado esperas en un archivo con 1 782 contigs? ¿Y cuántas líneas de datos?

In [ ]:
lineas <- readLines(gzfile(VCF))
es_encabezado <- startsWith(lineas, "##")

c(lineas_totales = length(lineas),
  lineas_de_encabezado = sum(es_encabezado),
  lineas_de_datos = sum(!startsWith(lineas, "#")))

head(lineas[es_encabezado], 6)      # el encabezado empieza describiendo el formato

In [ ]:
# La fila que nombra las columnas y la primera línea de datos (recortadas para que quepan)
cat(substr(lineas[startsWith(lineas, "#CHROM")], 1, 180), "\n\n")
cat(substr(lineas[!startsWith(lineas, "#")][1], 1, 260), "\n")

**Léelo con calma.** En la primera línea de datos identifica: el contig y la posición, el alelo de referencia (`REF`) y el alternativo (`ALT`), cuántas muestras hay y qué campos aparecen en `FORMAT`.

En este archivo `FORMAT` es siempre el mismo: **`GT:AD:DP:GQ`**.

- `GT` — el genotipo: `0/0` (homocigoto referencia), `0/1` (heterocigoto), `1/1` (homocigoto alternativo), `./.` (sin dato).
- `AD` — lecturas a favor del alelo referencia y del alternativo, en ese orden.
- `DP` — profundidad total del genotipo (cuántas lecturas lo sostienen).
- `GQ` — calidad del genotipo (*genotype quality*), en escala phred.

> **Ojo con el `INFO`.** Campos como `AC`, `AF` y `AN` son conteos sobre el archivo **original** de 502 muestras. Al hacer un subconjunto hay que **recalcularlos**; si no, quedan mintiendo. En este archivo ya vienen recalculados.

## 2. Leer con `vcfR` y construir la matriz de dosis

`vcfR` separa las partes del archivo. Luego convertimos cada genotipo en un número: el **número de copias del alelo alternativo** (`0/0` → 0, `0/1` → 1, `1/1` → 2, `./.` → `NA`).

In [ ]:
vcf <- read.vcfR(VCF, verbose = FALSE)
vcf                                                     # resumen del objeto
getFIX(vcf)[1:3, c("CHROM", "POS", "REF", "ALT")]       # las columnas fijas de los 3 primeros SNP
unique(vcf@gt[, "FORMAT"])                              # ¿qué trae cada genotipo?

In [ ]:
snp_id <- paste(getCHROM(vcf), getPOS(vcf), sep = "_")    # nombre del SNP: contig_posición
gt <- extract.gt(vcf, element = "GT")                     # matriz SNP x árbol
rownames(gt) <- snp_id
table(gt, useNA = "ifany")                                # ¿qué genotipos existen?

Fíjate en dos cosas de la tabla anterior: aparecen genotipos **faseados** (`0|0`, `0|1`, `1|1`) y una columna grande de `NA`.

El `|` significa que el programa se atrevió a asignar cada alelo a un cromosoma; el `/` significa que no. Para calcular frecuencias da lo mismo: `0/1` y `0|1` son el mismo heterocigoto. Por eso la función siguiente acepta las dos formas.

In [ ]:
a_dosis <- function(gt) {
  dos <- matrix(NA_integer_, nrow(gt), ncol(gt), dimnames = dimnames(gt))
  dos[gt %in% c("0/0", "0|0")] <- 0L
  dos[gt %in% c("0/1", "1/0", "0|1", "1|0")] <- 1L
  dos[gt %in% c("1/1", "1|1")] <- 2L
  dos
}
G <- t(a_dosis(gt))                  # filas = árboles, columnas = SNP

meta <- read.csv(META, stringsAsFactors = FALSE)
stopifnot(all(c("id", "sitio_n") %in% names(meta)))
meta <- meta[match(rownames(G), meta$id), ]
stopifnot(all(meta$id == rownames(G)))   # ¡siempre verificar el orden!

dim(G)                               # árboles x SNP
table(meta$sitio_n, useNA = "ifany") # árboles por sitio (NA = sin ficha)
G[1:5, 1:6]

**Regla de oro.** Si `meta` estuviera en otro orden, todo se calcularía sin error y estaría mal: el árbol 1 tendría el sitio del árbol 7. Por eso la verificación `stopifnot()`: si el orden no calza, el notebook se detiene en vez de mentir.

En la tabla de sitios apareció un `NA`: hay **muestras del VCF que no tienen ficha** en el material publicado. No es un error del material, es lo que pasa con los datos reales. Las contamos y las resolvemos más adelante.

In [ ]:
sin_ficha <- rownames(G)[is.na(meta$sitio_n)]
length(sin_ficha)
sin_ficha

Anota esos nombres: son 7 muestras que el archivo trae pero el material complementario del estudio no explica. En §6 vas a descubrir qué son las seis primeras.

## 3. La ficha técnica del archivo

Una **ficha técnica** resume lo esencial del archivo en una tabla. Es lo primero que se escribe al recibir datos y lo que después se cita en los Métodos.

Antes de calcularla, fijemos la notación: llamamos **p** a la frecuencia del **alelo alternativo** (el `AF` del VCF), de modo que `p = media de la dosis / 2`. La **MAF** —frecuencia del alelo **menor**— es `min(p, 1 − p)`: no es lo mismo que p.

In [ ]:
falt_ind <- rowMeans(is.na(G))                 # fracción de faltantes por árbol
falt_snp <- colMeans(is.na(G))                 # fracción de faltantes por SNP
p   <- colMeans(G, na.rm = TRUE) / 2           # frecuencia del alelo alternativo
maf <- pmin(p, 1 - p)                          # frecuencia del alelo menor
polimorfico <- !is.na(maf) & maf > 0           # ¿el SNP varía entre los árboles?

genotipos_validos <- c("0/0", "0|0", "0/1", "1/0", "0|1", "1|0", "1/1", "1|1")
ficha <- data.frame(
  campo = c("Muestras", "SNP", "Contigs distintos", "Sitios", "Árboles por sitio (mín-máx)",
            "Genotipos faltantes (%)", "Muestra con más faltantes (%)",
            "SNP con algún faltante (%)", "SNP sin ningún genotipo", "SNP sin variación", "SNP con 0 < MAF < 0,05",
            "SNP con MAF >= 0,05", "Genotipos con otro formato (no 0/0, 0/1, 1/1, ./.)"),
  valor = c(nrow(G), ncol(G), length(unique(getCHROM(vcf))),
            length(unique(na.omit(meta$sitio_n))),
            paste(range(table(meta$sitio_n)), collapse = "-"),
            round(100 * mean(is.na(G)), 1), round(100 * max(falt_ind), 1),
            round(100 * mean(falt_snp > 0), 1), sum(is.na(maf)), sum(!polimorfico),
            sum(polimorfico & maf < 0.05), sum(polimorfico & maf >= 0.05),
            sum(!is.na(gt) & !(gt %in% genotipos_validos))))
ficha

In [ ]:
meta$grupo <- ifelse(is.na(meta$sitio_n), "sin ficha", paste("sitio", meta$sitio_n))
d_ind <- data.frame(id = rownames(G), grupo = meta$grupo, faltantes = 100 * falt_ind)

ggplot(d_ind, aes(reorder(id, faltantes), faltantes, fill = grupo)) +
  geom_col(show.legend = FALSE) + coord_flip() +
  labs(x = NULL, y = "% de genotipos faltantes") +
  theme(axis.text.y = element_text(size = 5))

ggplot(data.frame(maf, polimorfico)[is.finite(maf), ], aes(maf, fill = polimorfico)) +
  geom_histogram(bins = 30) +
  labs(x = "MAF (frecuencia del alelo menor)", y = "N° de SNP", fill = "¿Varía?")

**Discute.**

- ¿Coincide la ficha con tus predicciones?
- ¿Qué árbol destaca por sus faltantes y qué harías con él?
- En el gráfico de barras se ve quién es quién: ¿los que pierden más datos están agrupados por sitio, o son casos aislados?
- Mira el histograma de MAF: la primera barra son los **monomórficos** (MAF = 0). ¿Cuántos SNP aportan poca información?
- **Compara con el archivo completo:** este subconjunto tiene 207 muestras y en él 497 de los 1 782 SNP no varían. En el archivo completo de 502 árboles solo 53 SNP eran monomórficos de los mismos 1 782. **La MAF depende del tamaño de muestra**: filtrar por MAF con pocos individuos elimina variantes que sí existen.

## 4. Primera señal de alerta: exceso de heterocigotos

Un SNP en el que **casi todos** los árboles son heterocigotos no se comporta como un locus mendeliano simple: suele ser un **parálogo** (dos copias del genoma colapsadas en un contig) o una duplicación. Si se queda, infla la heterocigosis y hunde la Fis.

El método HDplot, usado en el estudio original, descarta los SNP con una proporción de heterocigotos **H mayor que 0,6**. Pero antes de aplicar el umbral, veamos cuánto puede subir H **solo por azar**.

In [ ]:
het_snp <- colMeans(G == 1, na.rm = TRUE)       # H: proporción de heterocigotos por SNP

ggplot(data.frame(het_snp = het_snp[!is.na(het_snp)]), aes(het_snp)) +
  geom_histogram(bins = 30, fill = "darkorange") +
  geom_vline(xintercept = 0.6, lty = 2, colour = "red") +
  labs(x = "Proporción de heterocigotos por SNP (H)", y = "N° de SNP")

sospechosos <- names(which(het_snp > 0.6))
length(sospechosos)
head(sospechosos, 5)

In [ ]:
# ¿Cuánto puede subir H solo por azar? Un SNP "normal" con 50 % de heterocigotos esperados:
# probabilidad de que, por azar, más del 60 % de los árboles salgan heterocigotos
n_arboles <- c(6, 10, 25, 50, 100, 207, 502)
data.frame(arboles = n_arboles,
           prob_azar_H_mayor_0.6 = round(1 - pbinom(floor(0.6 * n_arboles), n_arboles, 0.5), 5))
# Con pocos árboles, H salta en escalones de 1/n: por eso la probabilidad no baja de forma monótona.

**Idea clave.** Con 6 o 10 árboles, un SNP perfectamente normal supera H = 0,6 con una probabilidad nada despreciable; con 207, casi nunca. Por eso un filtro de parálogos funciona en un conjunto grande, pero en un recorte pequeño solo sirve para **sospechar**.

La evidencia mejora cuando el patrón **se repite entre sitios**: un parálogo se comporta mal en todas las poblaciones; un exceso de heterocigotos por azar no.

In [ ]:
sitios <- sort(unique(na.omit(meta$sitio_n)))
H_por_sitio <- sapply(sitios, function(k)
  colMeans(G[which(meta$sitio_n == k), sospechosos, drop = FALSE] == 1, na.rm = TRUE))

n_sitios_H <- rowSums(H_por_sitio > 0.6, na.rm = TRUE)
table(n_sitios_H)                                  # ¿en cuántos sitios supera 0,6 cada sospechoso?
c(sospechosos = length(sospechosos),
  en_los_20_sitios = sum(n_sitios_H == length(sitios)))

**Discute.** ¿Cuántos sospechosos se comportan igual en los 20 sitios? Esos son los candidatos sólidos a parálogo. ¿Los eliminarías a todos, a esos solos, o a ninguno? ¿Qué evidencia adicional pedirías?

## 5. Segunda señal de alerta: profundidad y calidad del genotipo

Si un genotipo se apoya en **pocas lecturas**, el programa puede leer solo una de las dos copias de un heterocigoto y llamarlo homocigoto (*allelic dropout*). Si eso ocurre, la **proporción de heterocigotos debería ser menor cuando la profundidad es baja**. Es una predicción comprobable.

In [ ]:
dp <- t(extract.gt(vcf, element = "DP", as.numeric = TRUE))   # filas = árboles
gq <- t(extract.gt(vcf, element = "GQ", as.numeric = TRUE))
stopifnot(identical(dim(dp), dim(G)))

print(summary(as.vector(dp)))
print(summary(as.vector(gq)))
cat("Genotipos LLAMADOS con DP < 8 :", round(100 * mean(dp < 8, na.rm = TRUE), 1), "%\n")
cat("Genotipos LLAMADOS con GQ < 20:", round(100 * mean(gq < 20, na.rm = TRUE), 1), "%\n")

In [ ]:
llamado <- !is.na(G)                                  # solo donde hubo genotipo
d <- data.frame(dp = as.vector(dp)[as.vector(llamado)],
                g  = as.vector(G)[as.vector(llamado)])
d$rango_dp <- cut(d$dp, c(-1, 3, 7, 11, 19, Inf),
                  labels = c("0-3", "4-7", "8-11", "12-19", "20+"))
tab <- data.frame(genotipos = tapply(d$g, d$rango_dp, length),
                  prop_heterocigotos = round(tapply(d$g == 1, d$rango_dp, mean), 3))
tab$rango_dp <- rownames(tab)
print(tab, row.names = FALSE)

ggplot(tab, aes(rango_dp, prop_heterocigotos)) + geom_col(fill = "steelblue") +
  labs(x = "Profundidad (DP) del genotipo", y = "Proporción de heterocigotos")

**Discute.** ¿Sube la proporción de heterocigotos al aumentar la profundidad? ¿Qué le pasaría a Ho (y a Fis) si no filtráramos por profundidad? Esa es la razón técnica del umbral **DP ≥ 8** del estudio.

Ahora midamos lo que cuesta aplicar el filtro completo del estudio: **DP ≥ 8 y GQ ≥ 20**.

In [ ]:
conserva <- dp >= 8 & gq >= 20 & llamado             # TRUE = el genotipo se mantiene
cat("Genotipos llamados:", sum(llamado), "\n")
cat("Se convierten en faltante:", round(100 * (1 - sum(conserva) / sum(llamado)), 1), "%\n\n")

falt_final <- 100 * (1 - rowSums(conserva) / ncol(G))
cat("Faltantes finales por árbol: mediana", round(median(falt_final), 1),
    "% | mínimo", round(min(falt_final), 1), "% | máximo", round(max(falt_final), 1), "%\n")
cat("Árboles que quedan con más de 30 % de faltantes:", sum(falt_final > 30), "de", nrow(G), "\n\n")

d_fin <- data.frame(id = rownames(G), grupo = meta$grupo, faltantes = falt_final)
ggplot(d_fin, aes(reorder(id, faltantes), faltantes, fill = grupo)) +
  geom_col(show.legend = FALSE) + coord_flip() +
  labs(x = NULL, y = "% de genotipos faltantes después de DP >= 8 y GQ >= 20") +
  theme(axis.text.y = element_text(size = 5))

**Idea clave.** El filtro por profundidad y calidad **no reparte la pérdida de forma pareja**: la mediana ronda el 30 % y hay árboles que quedan casi vacíos. Eso arrastra una cadena de consecuencias: menos datos por árbol → el filtro de faltantes por individuo (Sección 1) elimina muestras → cambia el n por sitio → cambia He y Fis.

Por eso un filtro no se elige por «lo que se ve bien», sino midiendo qué se pierde. Y cada decisión se reporta.

## 6. Tercera señal de alerta: réplicas

En el archivo publicado hay árboles genotipados dos veces (réplicas técnicas). No son árboles distintos: si se quedan, **duplican** un individuo, inflan el n y sesgan las frecuencias. Y hay un bono: la discrepancia entre las dos copias del mismo árbol **mide el error de genotipado** del estudio.

¿Cómo se detectan sin mirar los nombres? Comparando genotipos: dos réplicas del mismo árbol coinciden en casi todos los SNP, mucho más que dos árboles distintos.

In [ ]:
# Proporción de genotipos idénticos entre cada par de muestras (identidad por estado)
A <- G; A[is.na(A)] <- 0
ident <- matrix(NA_real_, nrow(G), nrow(G), dimnames = list(rownames(G), rownames(G)))

for (i in seq_len(nrow(G))) {
  ambos   <- !is.na(G) & matrix(!is.na(G[i, ]), nrow(G), ncol(G), byrow = TRUE)
  iguales <- (A == matrix(A[i, ], nrow(G), ncol(G), byrow = TRUE)) & ambos
  ident[i, ] <- rowSums(iguales) / pmax(rowSums(ambos), 1)
  ident[i, rowSums(ambos) < 500] <- NA      # pares con muy pocos SNP comparables no cuentan
}
diag(ident) <- NA

pares <- which(ident > 0.95, arr.ind = TRUE)
pares <- pares[pares[, 1] < pares[, 2], , drop = FALSE]     # sin repetir ni la diagonal
cat("Pares con identidad > 0,95:", nrow(pares), "\n\n")

resumen <- data.frame(
  muestra_1 = rownames(G)[pares[, 1]],
  muestra_2 = rownames(G)[pares[, 2]],
  identidad = round(ident[pares], 4)
)
resumen$discrepancia_pct <- round(100 * (1 - resumen$identidad), 2)
print(resumen, row.names = FALSE)

cat("\n¿Cuántas de esas muestras no tenían ficha?", 
    sum(is.na(meta$sitio_n[match(c(resumen$muestra_1, resumen$muestra_2), rownames(G))])), "de",
    2 * nrow(resumen), "\n")

**Idea clave.** Las **6 muestras sin ficha** que aparecieron en §3 son las parejas de esos seis pares: en el nombre IGA llevan una `R` (`66-R253`, `107-R234`, …). El nombre lo insinúa; la identidad genotípica lo **demuestra**.

Esa discrepancia de 2–4 % es la estimación del error de genotipado del estudio. Y hay una advertencia metodológica: la identidad entre árboles distintos también es alta (≈0,93), porque la mayoría de los SNP son homocigotos referencia en todos. Por eso el umbral 0,95 solo funciona porque las réplicas están claramente arriba.

**Decisión:** se conserva una copia de cada par (la que tenga mejor calidad) y se anota la discrepancia en los Métodos.

## 7. Diversidad por sitio con datos reales: Ho, He y Fis

Con el archivo ya limpio (filtrado por DP y GQ, sin réplicas y sin la muestra vacía) recién ahora tiene sentido calcular diversidad. Recordemos las tres definiciones de la Sección 2:

- **Ho** — heterocigosidad observada: la proporción de genotipos heterocigotos que hay en los datos.
- **He** — heterocigosidad esperada bajo Hardy-Weinberg: `2pq`, con la corrección `2n/(2n−1)` para muestras chicas.
- **Fis** — `1 − Ho/He`. Vale 0 si el sitio está en equilibrio; **positivo** si hay déficit de heterocigotos (endogamia, subdivisión); **negativo** si hay **exceso**.

Con datos reales hay una diferencia importante respecto de la Sección 2: acá no hay una «verdad» que comparar, y Ho y He dependen del conjunto de SNP que uses. Vamos a verlo.

In [ ]:
# ---- el archivo limpio ----
G_limpio <- G
malo <- !(!is.na(G) & dp >= 8 & gq >= 20)      # genotipo a descartar
G_limpio[malo] <- NA

replicas <- c("66-R253-A-B09", "107-R234-B-C02", "108-R235-B-D02",
              "407-R141-E-G03", "346-R-285-D-B08", "183-R406-B-G11")
quitar <- rownames(G) %in% replicas | rowMeans(is.na(G)) > 0.9
G_limpio <- G_limpio[!quitar, ]
meta_limpio <- meta[!quitar, ]

dim(G_limpio)                              # árboles x SNP después de limpiar
round(100 * mean(is.na(G_limpio)), 1)      # % de faltantes que queda

p_limpio <- colMeans(G_limpio, na.rm = TRUE) / 2
maf_limpio <- pmin(p_limpio, 1 - p_limpio)
pol_limpio <- !is.na(maf_limpio) & maf_limpio > 0
sum(pol_limpio)                            # SNP que todavía varían

In [ ]:
diversidad_por_sitio <- function(G, sitio) {
  sitio <- factor(sitio)
  do.call(rbind, lapply(levels(sitio), function(k) {
    Gk <- G[sitio == k, , drop = FALSE]
    n  <- colSums(!is.na(Gk)); p <- colMeans(Gk, na.rm = TRUE) / 2
    ok <- n > 1
    he <- ((2 * n / (2 * n - 1)) * 2 * p * (1 - p))[ok]      # He insesgada
    ho <- colMeans(Gk == 1, na.rm = TRUE)[ok]
    data.frame(sitio_n = k, n = nrow(Gk), Ho = mean(ho), He = mean(he),
               Fis = 1 - mean(ho) / mean(he), stringsAsFactors = FALSE)
  }))
}

div <- diversidad_por_sitio(G_limpio[, pol_limpio], meta_limpio$sitio_n)
div$lat   <- tapply(meta_limpio$lat, meta_limpio$sitio_n, mean)[div$sitio_n]
# el nombre del sitio no se promedia: se toma el primero de cada sitio
div$sitio <- tapply(meta_limpio$sitio, meta_limpio$sitio_n,
                    function(x) unique(x[!is.na(x)])[1])[div$sitio_n]
div <- div[, c("sitio_n", "sitio", "lat", "n", "Ho", "He", "Fis")]
div[5:7] <- round(div[5:7], 3)
print(div, row.names = FALSE)
cat("\nHo medio:", round(mean(div$Ho), 3), "| He medio:", round(mean(div$He), 3),
    "| Fis medio:", round(mean(div$Fis), 3), "\n")

In [ ]:
ggplot(div, aes(He, Ho, colour = lat)) + geom_abline(lty = 2, colour = "grey50") +
  geom_point(size = 3.2) + geom_text(aes(label = sitio_n), vjust = -0.9, size = 3) +
  scale_colour_viridis_c() +
  labs(x = "He (esperada)", y = "Ho (observada)", colour = "latitud") + coord_equal()

ggplot(div, aes(reorder(sitio, lat), Fis)) + geom_col(fill = "firebrick") +
  geom_hline(yintercept = 0) +
  labs(x = "sitio (ordenado por latitud, de norte a sur)", y = "Fis")

**Discute.** Los 20 sitios tienen **Fis negativo** (≈ −0,25 a −0,35): hay más heterocigotos de los que Hardy-Weinberg predice, y bastante más de lo que explica el azar.

Antes de buscar explicaciones biológicas, revisemos una técnica: el filtro DP/GQ **no** lo arregló (lo empeoró un poco). Si el problema no es la profundidad, ¿qué más puede producir un exceso de heterocigotos? Volvamos a la sección 4: los SNP con H > 0,6, los sospechosos de ser **parálogos** (dos copias del genoma colapsadas; el «heterocigoto» que se ve es en realidad un sitio que existe dos veces).

In [ ]:
het_limpio <- colMeans(G_limpio == 1, na.rm = TRUE)

resumen_filtros <- do.call(rbind, lapply(c(1, 0.6, 0.5, 0.4), function(umbral) {
  sel <- pol_limpio & (het_limpio <= umbral | is.na(het_limpio))
  dv  <- diversidad_por_sitio(G_limpio[, sel], meta_limpio$sitio_n)
  data.frame(H_max = umbral, SNP = sum(sel), Ho = mean(dv$Ho), He = mean(dv$He),
             Fis_medio = mean(dv$Fis), Fis_min = min(dv$Fis), Fis_max = max(dv$Fis))
}))
resumen_filtros[3:7] <- round(resumen_filtros[3:7], 3)
print(resumen_filtros, row.names = FALSE)

**Idea clave.** Al quitar los SNP con H > 0,6, la Fis media pasa de ≈ −0,29 a ≈ −0,05; con el corte en H > 0,4 queda en ≈ 0,00 y Ho ≈ He. O sea: **el exceso de heterocigotos era de los parálogos**, no de la biología de la lenga. Ese es el argumento del estudio para aplicar HDplot, y es la respuesta a la alerta 1 de la ficha técnica.

Nota metodológica: Ho y He también cambian según qué SNP dejes. Con todas las variantes Ho ≈ 0,15 y He ≈ 0,11; si exiges MAF ≥ 0,05 (solo SNP comunes) suben a Ho ≈ 0,44 y He ≈ 0,33. Ninguno de los dos números es «el» valor de la diversidad: siempre se reporta con el conjunto y los filtros que lo produjeron.

## 8. ¿Se separan los sitios? PCA de los árboles

El PCA resume cientos de SNP en dos ejes que concentran la mayor variación. No es un test ni una prueba de estructura: es una **foto** para mirar los datos. Cada árbol se vuelve un punto; si dos sitios caen en zonas distintas del plano, hay diferenciación genética.

Con datos reales hay dos detalles prácticos: hay datos faltantes (los imputamos con la media del SNP) y los primeros componentes suelen explicar **poco** porcentaje, porque hay muchos SNP raros distintos entre individuos.

In [ ]:
Gi <- G_limpio[, pol_limpio]
mu <- colMeans(Gi, na.rm = TRUE)
for (j in seq_len(ncol(Gi))) Gi[is.na(Gi[, j]), j] <- mu[j]      # imputar con la media

pc <- prcomp(Gi, center = TRUE, scale. = FALSE)
ve <- round(100 * pc$sdev^2 / sum(pc$sdev^2), 1)
ve[1:4]

d_pc <- data.frame(meta_limpio, PC1 = pc$x[, 1], PC2 = pc$x[, 2])
ggplot(d_pc, aes(PC1, PC2, colour = factor(sitio_n))) + geom_point(size = 2.8) +
  labs(x = paste0("PC1 (", ve[1], " %)"), y = paste0("PC2 (", ve[2], " %)"), colour = "sitio")

In [ ]:
pc1_sitio <- tapply(pc$x[, 1], meta_limpio$sitio_n, mean)
d_pc1 <- data.frame(sitio_n = names(pc1_sitio), PC1 = as.numeric(pc1_sitio),
                    lat = tapply(meta_limpio$lat, meta_limpio$sitio_n, mean)[names(pc1_sitio)],
                    sitio = tapply(meta_limpio$sitio, meta_limpio$sitio_n,
                                   function(x) unique(x[!is.na(x)])[1])[names(pc1_sitio)])

ggplot(d_pc1, aes(lat, PC1)) + geom_point(size = 3) +
  geom_text(aes(label = sitio_n), vjust = -0.9, size = 3) +
  labs(x = "latitud (°S)", y = "PC1 medio del sitio")

c(varianza_PC1_pct = ve[1], cor_PC1_latitud = round(cor(d_pc1$PC1, d_pc1$lat), 2))

**Discute.** El PC1 explica apenas **3–4 %** de la variación y, sin embargo, ordena los sitios de norte a sur casi perfectamente (correlación ≈ −0,87 con la latitud: Epulaufquen en un extremo, Ushuaia en el otro, los sitios de ~43° S en el medio). Tres ideas que valen para todo el curso:

- **Poca varianza explicada no significa «no hay estructura»**: el PCA reparte la variación entre muchos ejes porque hay muchos SNP raros; lo que importa es el **patrón** de los individuos en el plano.
- La meseta del centro (43–44° S) y el salto hacia el norte coinciden con la **ruptura filogeográfica descrita para la lenga alrededor de los 42° S** (Mattera et al. 2020). Es una pista coherente, no una demostración.
- La diversidad es pareja (He ≈ 0,11 en todos los sitios) y la diferenciación aparece en el PCA: son dos caras distintas. ¿Cuál usarías para decir «estas poblaciones están diferenciadas», y con qué estadístico lo demostrarías?

---

## Resumen de la parte demostrativa

1. Un VCF se lee por partes: encabezado, columnas fijas, `FORMAT` y un bloque por árbol.
2. Se convierte en una matriz de dosis y **siempre** se comprueba el orden contra los metadatos.
3. La ficha técnica resume faltantes, SNP sin variación, muestras sin ficha y MAF antes de analizar nada.
4. Un H alto sugiere parálogos, pero con pocos árboles también ocurre por azar: la evidencia mejora si se repite entre sitios.
5. Con poca profundidad se pierden heterocigotos: por eso se filtra por DP y GQ, midiendo lo que se pierde.
6. Las réplicas se detectan por identidad genotípica y su discrepancia estima el error de genotipado.
7. Con el archivo **sin filtrar**, Ho sale mayor que He (Fis negativo): la señal de que hay parálogos y errores todavía adentro.
8. Sobre el archivo limpio, Ho ≈ 0,15 y He ≈ 0,11, con **Fis negativo en los 20 sitios** (−0,25 a −0,35).
9. Quitar los SNP con H > 0,6 lleva la Fis media a ≈ −0,05; con el corte en 0,4 queda en ≈ 0,00 y Ho ≈ He. El exceso de heterocigotos **era de los parálogos**.
10. El **PCA** ordena los sitios de norte a sur con solo 3–4 % de la varianza en el PC1: poca varianza explicada no es ausencia de estructura.

> Usamos las mismas herramientas de la Sección 2 (Ho, He, Fis, PCA), pero ahora sobre datos reales y con un archivo que ya pasó por tres decisiones de filtrado. Ahí la conclusión cambia: no hay un «valor verdadero» de diversidad, hay uno por cada conjunto de SNP y cada filtro, y hay que reportarlo.

---
# Parte de ejercicios

Resuelve en las celdas siguientes. Anota tus **decisiones** (umbrales y por qué): serán tu sección de *Métodos*. Los ejercicios usan el mismo archivo real de la parte demostrativa.

### E1. Completa la ficha técnica
A la ficha de la sección 3 agrega cuatro campos: **SNP con más de 20 % de faltantes**, **árboles con más de 50 % de faltantes**, **SNP con H mayor que 0,6** y **muestras sin ficha**. ¿Alguno te preocupa?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: sum(falt_snp > 0.2), sum(falt_ind > 0.5), sum(het_snp > 0.6, na.rm = TRUE), sum(is.na(meta$sitio_n))

> **Idea clave:** una ficha técnica es útil solo si se compara con un criterio; por eso cada campo se mira contra el umbral que pretendes usar.

### E2. Tabla de sensibilidad de filtros
Construye una tabla con el **número de SNP que quedan** al combinar un umbral máximo de faltantes por SNP (0, 0,1, 0,2 y 0,3) con una MAF mínima (0, 0,05 y 0,10), contando solo SNP que varían. ¿Qué combinación elegirías, y por qué? ¿Qué pasa con los SNP si exiges MAF ≥ 0,10 con solo 207 muestras y variantes raras de secuenciación dirigida?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: umbrales <- expand.grid(faltantes = c(0, 0.1, 0.2, 0.3), maf_min = c(0, 0.05, 0.10))
#        y para cada fila: sum(falt_snp <= faltantes & polimorfico & maf >= maf_min)

> **Idea clave:** no hay umbral «correcto», hay umbrales **justificados y reportados**. Y la MAF mínima elimina mucho más de lo que uno espera: la mayoría de las variantes de un estudio real son raras.

### E3. ¿Son parálogos de verdad?
Toma los SNP sospechosos (H > 0,6). Para cada sitio por separado, calcula H de esos SNP. ¿Cuáles tienen H alto **en los 20 sitios**? ¿Y en al menos 15? ¿Los eliminarías? Justifica con lo que viste en la sección 4.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: ya tienes H_por_sitio y n_sitios_H de la sección 4; prueba nombres(n_sitios_H)[n_sitios_H >= 15]

> **Idea clave:** un solo criterio (H > 0,6) en un recorte pequeño solo permite **sospechar**; la evidencia mejora cuando el patrón se repite entre sitios.

### E4. Los filtros del artículo sobre datos reales
Con las matrices `dp` y `gq`, aplica los umbrales del estudio: conserva un genotipo solo si **DP ≥ 8 y GQ ≥ 20**. ¿Qué porcentaje de genotipos llamados se convierte en dato faltante? ¿Qué árboles pierden más? Haz un gráfico de barras por árbol y decide cuántos árboles eliminarías después (con qué umbral de faltantes totales).

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: conserva <- dp >= 8 & gq >= 20 & !is.na(G)
#        perdida_por_arbol <- 100 * (1 - rowSums(conserva) / ncol(G))

> **Idea clave:** el filtro por profundidad y calidad no reparte la pérdida de forma pareja entre árboles: algunos quedan con muy pocos genotipos y terminan eliminados por el filtro de faltantes.

### E5. Encuentra las réplicas tú misma
Sin mirar la sección 6: calcula, para cada par de muestras, la proporción de genotipos **idénticos** con los 1 782 SNP y busca los pares con más de 95 % de coincidencia. Luego calcula la **discrepancia** de cada par: esa cifra estima el error de genotipado. ¿Coinciden tus seis pares con los que encontró la demostración? ¿Qué umbral separa limpiamente réplicas de árboles distintos?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: A <- G; A[is.na(A)] <- 0      y compara cada fila con todas las demás
# Pista 2: ojo con los pares que comparten pocos SNP comparables (el árbol casi vacío)
# Pista 3: mira la distribución de identidades: ¿dónde está el «valle» entre réplicas y no réplicas?

> **Idea clave:** las réplicas no son árboles distintos. Hay que detectarlas y quitar una copia antes de calcular frecuencias, y su discrepancia mide el error de genotipado.

### E6. Tres alertas y una decisión
Con lo que encontraste, completa la tabla. Una alerta por fila; cada decisión debe incluir **un umbral y su justificación**.

### E7. El umbral de parálogos cambia la Fis
Repite la tabla de la sección 7 con cortes de H en 0,7 / 0,6 / 0,5 / 0,4 / 0,3. Grafica `Fis_medio` contra el umbral. ¿Dónde se estabiliza? ¿Qué umbral elegirías y qué pierdes al bajarlo tanto (mira la columna `SNP`)?


| # | Alerta (qué viste y dónde) | Posible causa | Decisión (con umbral) | Justificación |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

---
## Errores y dudas frecuentes

- **«No encuentra `datos_ejemplo/lenga_real_1782snp.vcf.gz`»** → abriste Jupyter desde otra carpeta; ábrelo desde la carpeta del repositorio.
- **«`readLines` falla o lee basura»** → el archivo está comprimido: usa `readLines(gzfile(VCF))`.
- **«Me sale `NA` en `maf`»** → un SNP sin ningún genotipo; no tiene frecuencia calculable.
- **«`table(gt)` muestra `0|0` y `0/1`»** → hay genotipos faseados; `a_dosis()` ya acepta las dos formas.
- **«Faltan 7 filas en los metadatos»** → son las muestras sin ficha del archivo publicado: 6 réplicas y una muestra casi vacía. Se resuelven en §6, no rellenando con ceros.
- **«`extract.gt()` me devuelve SNP × árboles»** → sí: transpón (`t()`) antes de comparar con `G`, que es árboles × SNP.
- **«`G_limpio[malo] <- NA` da error de `NA`»** → usa una condición que nunca sea `NA`, como `!(!is.na(G) & dp >= 8 & gq >= 20)`.
- **«La Fis sale muy negativa»** → es el exceso de heterocigotos por parálogos: revisa §7 y quita los SNP con H alto.
- **«El PCA explica solo un 3 %»** → es normal con miles de SNP raros; mira el patrón y la correlación con la latitud, no el porcentaje.

## Tarea (se revisa al comienzo de la próxima sección)

1. Entrega tu **ficha técnica** completa (E1) y la tabla de **tres alertas** (E6).
2. Escribe en **5 líneas** los umbrales de filtrado que usarás y su justificación: será el borrador de tu sección de *Métodos*. Incluye al menos: faltantes por SNP, MAF mínima, DP, GQ y qué haces con las réplicas.
3. Responde en dos frases: **¿por qué no se puede decidir el umbral de MAF sin saber el tamaño de muestra?**
4. Interpreta el **PCA** en tres líneas: qué muestra, qué no permite concluir, y qué estadístico pedirías para afirmar que los sitios están diferenciados.

**Para profundizar:** [Introducción al VCF (Browning)](https://faculty.washington.edu/browning/beagle/intro-to-vcf.html) · [Visualizar un VCF con vcfR](https://knausb.github.io/vcfR_documentation/visualization_2.html) · [Datos de lenga en Zenodo](https://zenodo.org/records/10018374) · [Preprint de Sekely et al.](https://www.biorxiv.org/content/10.1101/2023.04.28.538677v1)